# Tool Calling (함수 호출) 실습

LLM은 직접 날씨를 조회하거나 계산을 수행할 수 없다.  
**Tool Calling**은 LLM이 외부 함수가 필요하다고 판단하면 함수 이름과 인자를 JSON으로 반환하고,  
우리가 실제 함수를 호출한 뒤 결과를 다시 LLM에 넘기는 구조다.

```
사용자 질문
    ↓
LLM → "get_weather({city: '서울'})를 호출해"
    ↓
우리 코드가 실제 함수 호출
    ↓
결과를 LLM에 전달 → 최종 답변 생성
```

## Part 1. Python 함수 정의

실제로 호출될 Python 함수를 먼저 만든다.  
실습에서는 실제 API 대신 더미 데이터를 반환한다.

In [1]:
import json

def get_weather(city: str) -> str:
    dummy_data = {
        "서울": {"temperature": 22, "condition": "맑음", "humidity": 45},
        "부산": {"temperature": 25, "condition": "흐림", "humidity": 70},
        "제주": {"temperature": 27, "condition": "비",   "humidity": 85},
    }
    result = dummy_data.get(city, {"temperature": 20, "condition": "알 수 없음", "humidity": 50})
    return json.dumps({"city": city, **result}, ensure_ascii=False)

available_functions = {"get_weather": get_weather}

print(get_weather("서울"))
print(get_weather("부산"))

{"city": "서울", "temperature": 22, "condition": "맑음", "humidity": 45}
{"city": "부산", "temperature": 25, "condition": "흐림", "humidity": 70}


## Part 2. 스키마 작성 실습: `get_weather`

Python 함수 정의와 짝을 이루는 스키마는 다음과 같이 작성한다.

### `tools` 리스트의 구조

```python
tools = [
    {
        "type": "function",
        "function": {
            # ① 함수 이름 – LLM이 "이 함수를 호출해"라고 지정할 때 사용
            "name": "get_weather",
            # ② 함수 설명 – LLM이 "이 함수를 언제 써야 하는지" 판단하는 핵심 단서
            "description": "지정된 도시의 현재 날씨 정보를 조회합니다",
            # ③ 파라미터 정의 – 이 함수에 뭘 넘겨야 하는지
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시명 (예: 서울, 부산)"
                    }
                },
                "required": ["city"]    # 필수 파라미터
            }
        }
    }
]
```

## Part 3. Gemini API로 Tool Calling

Gemini API는 `google-genai` 패키지를 사용한다.  
스키마 형식은 `types.FunctionDeclaration`으로 작성하며, 전체 흐름은 Ollama와 동일하다.

```
1단계: 사용자 질문 + tools → Gemini
2단계: Gemini → function_call (함수 이름 + 인자) 반환
3단계: 우리가 Python 함수 실행
4단계: 실행 결과를 Gemini에 전달 → 최종 답변 생성
```

In [2]:
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-2.5-flash"

# Gemini 스키마 — types.FunctionDeclaration으로 정의
get_weather_decl = types.FunctionDeclaration(
    name="get_weather",
    description="지정된 도시의 현재 날씨 정보를 조회합니다",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "city": types.Schema(
                type=types.Type.STRING,
                description="날씨를 조회할 도시명 (예: 서울, 부산)"
            )
        },
        required=["city"]
    )
)

tools_gemini = [types.Tool(function_declarations=[get_weather_decl])]
print("Gemini 스키마 준비 완료")

Gemini 스키마 준비 완료


### 3-1. 1단계 — Gemini가 도구를 선택하게 하기

In [3]:
user_message = "서울의 날씨가 어때?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_message,
    config=types.GenerateContentConfig(tools=tools_gemini),
)

print("=== Gemini 응답 ===")
print(f"text          : {response.text}")
print(f"function_calls: {response.function_calls}")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 50.989703162s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

### 3-2. 2단계 — 함수를 실행하고 결과를 Gemini에 돌려주기

In [4]:
chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(tools=tools_gemini),
)

response = chat.send_message(user_message)

if response.function_calls:
    for fc in response.function_calls:
        print(f"[도구 호출] {fc.name}({dict(fc.args)})")
        fn_result = available_functions[fc.name](**dict(fc.args))
        print(f"[실행 결과] {fn_result}")

        final_response = chat.send_message(
            types.Part.from_function_response(
                name=fc.name,
                response={"output": fn_result},
            )
        )
    print(f"\n[최종 답변] {final_response.text}")
else:
    print(f"[도구 없이 답변] {response.text}")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 47.261361339s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

### 3-3. 전체 흐름을 함수로 묶기 (Gemini)

In [5]:
def run_tool_calling_gemini(user_query: str, tools, avail_fns) -> str:
    chat = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(tools=tools),
    )
    response = chat.send_message(user_query)

    if not response.function_calls:
        return response.text

    last_response = None
    for fc in response.function_calls:
        fn_result = avail_fns[fc.name](**dict(fc.args))
        last_response = chat.send_message(
            types.Part.from_function_response(
                name=fc.name,
                response={"output": fn_result},
            )
        )
    return last_response.text


for q in ["서울의 날씨가 어때?", "부산이랑 제주 중에 어디가 더 따뜻해?", "오늘 점심 뭐 먹을까?"]:
    print(f"\n{'='*60}\n질문: {q}\n{'='*60}")
    print(run_tool_calling_gemini(q, tools_gemini, available_functions))


질문: 서울의 날씨가 어때?


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 44.956978437s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

## Part 4. Ollama로 Tool Calling

로컬 Ollama 모델로 동일한 흐름을 실행한다.  
스키마 형식은 OpenAI 호환 딕셔너리를 사용한다.

In [6]:
import ollama
import json

tools_ollama = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "지정된 도시의 현재 날씨 정보를 조회합니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시명 (예: 서울, 부산)"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

print(json.dumps(tools_ollama, indent=2, ensure_ascii=False))

[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "지정된 도시의 현재 날씨 정보를 조회합니다",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "날씨를 조회할 도시명 (예: 서울, 부산)"
          }
        },
        "required": [
          "city"
        ]
      }
    }
  }
]


### 4-1. 1단계 — Ollama가 도구를 선택하게 하기

In [7]:
user_message = "서울의 날씨가 어때?"

response = ollama.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": user_message}],
    tools=tools_ollama,
)

print("=== LLM 응답 ===")
print(f"content   : {response.message.content}")
print(f"tool_calls: {response.message.tool_calls}")

=== LLM 응답 ===
content   : 
tool_calls: [ToolCall(function=Function(name='get_weather', arguments={'city': '서울'}))]


### 4-2. 2단계 — 함수를 실행하고 결과를 Ollama에 돌려주기

In [8]:
messages = [{"role": "user", "content": user_message}]

response = ollama.chat(model="qwen2.5:3b", messages=messages, tools=tools_ollama)

if response.message.tool_calls:
    messages.append(response.message)
    for tc in response.message.tool_calls:
        print(f"[도구 호출] {tc.function.name}({tc.function.arguments})")
        fn_result = available_functions[tc.function.name](**tc.function.arguments)
        print(f"[실행 결과] {fn_result}")
        messages.append({"role": "tool", "content": fn_result})

    final_response = ollama.chat(model="qwen2.5:3b", messages=messages)
    print(f"\n[최종 답변] {final_response.message.content}")
else:
    print(f"[도구 없이 답변] {response.message.content}")

[도구 호출] get_weather({'city': '서울'})
[실행 결과] {"city": "서울", "temperature": 22, "condition": "맑음", "humidity": 45}

[최종 답변] 현재 서울의 날씨는 맑고 습도가 45%인 22℃입니다.


### 4-3. 전체 흐름을 함수로 묶기 (Ollama)

In [9]:
def run_tool_calling_ollama(user_query: str, tools, avail_fns) -> str:
    messages = [{"role": "user", "content": user_query}]
    response = ollama.chat(model="qwen2.5:3b", messages=messages, tools=tools)

    if not response.message.tool_calls:
        return response.message.content

    messages.append(response.message)
    for tc in response.message.tool_calls:
        fn_result = avail_fns[tc.function.name](**tc.function.arguments)
        messages.append({"role": "tool", "content": fn_result})

    return ollama.chat(model="qwen2.5:3b", messages=messages).message.content


for q in ["서울의 날씨가 어때?", "부산이랑 제주 중에 어디가 더 따뜻해?", "오늘 점심 뭐 먹을까?"]:
    print(f"\n{'='*60}\n질문: {q}\n{'='*60}")
    print(run_tool_calling_ollama(q, tools_ollama, available_functions))


질문: 서울의 날씨가 어때?
현재 서울의 날씨는 맑고 습도가 45%인 22℃입니다.

질문: 부산이랑 제주 중에 어디가 더 따뜻해?
According to the information provided by my last tool run, Busan's current temperature is 25°C with a humidity of 70%, and the weather condition is clear skies (흐림 in Korean).

Now let's check Jeju:
Could you please provide me with the current weather conditions for Jeju Island?

질문: 오늘 점심 뭐 먹을까?
당신이 좋아하는 요리나 음식ジャン慢慢地 prefers certain types of cuisine. Could you please provide more information or specify your preferences for lunch? I can help suggest options based on that. If not, I could try to recommend a general lunch option.

For example, do you have any specific dietary restrictions or food preferences such as vegetarian, vegan, gluten-free, etc.?


## Part 5. 멀티 함수 등록 & 자동 선택

LLM은 사용자 질문을 보고 **어떤 함수를 호출할지 스스로 선택**한다.

In [10]:
# 새 함수 정의 — get_exchange_rate (더미 데이터)
def get_exchange_rate(currency: str) -> str:
    """USD 대비 특정 통화의 환율을 반환한다."""
    dummy_rates = {
        "KRW": 1380.5,
        "JPY": 149.8,
        "EUR": 0.92,
        "GBP": 0.79,
        "CNY": 7.24,
    }
    rate = dummy_rates.get(currency.upper(), None)
    if rate is None:
        return json.dumps({"error": f"{currency} 환율 데이터 없음"}, ensure_ascii=False)
    return json.dumps({"from": "USD", "to": currency.upper(), "rate": rate}, ensure_ascii=False)

print(get_exchange_rate("KRW"))
print(get_exchange_rate("JPY"))

{"from": "USD", "to": "KRW", "rate": 1380.5}
{"from": "USD", "to": "JPY", "rate": 149.8}


In [11]:
# Gemini — 멀티 함수 스키마 등록
get_exchange_rate_decl = types.FunctionDeclaration(
    name="get_exchange_rate",
    description="USD 대비 특정 통화의 현재 환율을 조회합니다",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "currency": types.Schema(
                type=types.Type.STRING,
                description="환율을 조회할 통화 코드 (예: KRW, JPY, EUR)"
            )
        },
        required=["currency"]
    )
)

# 두 함수를 하나의 Tool 객체에 묶어 등록
tools_multi_gemini = [
    types.Tool(function_declarations=[get_weather_decl, get_exchange_rate_decl])
]

# Ollama — 멀티 함수 스키마 등록
tools_multi_ollama = [
    tools_ollama[0],   # 기존 get_weather
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "USD 대비 특정 통화의 현재 환율을 조회합니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "currency": {
                        "type": "string",
                        "description": "환율을 조회할 통화 코드 (예: KRW, JPY, EUR)"
                    }
                },
                "required": ["currency"]
            }
        }
    }
]

# 디스패치 딕셔너리 — 함수명 → 실제 함수
available_functions_multi = {
    "get_weather": get_weather,
    "get_exchange_rate": get_exchange_rate,
}

print("멀티 함수 등록 완료:", list(available_functions_multi.keys()))

멀티 함수 등록 완료: ['get_weather', 'get_exchange_rate']


### 5-1. Gemini 멀티 함수 자동 선택 테스트

날씨 질문, 환율 질문, 일반 잡담 세 가지로 LLM이 올바른 함수를 선택하는지 확인한다.

In [13]:
queries = [
    "서울 날씨 어때?",           # → get_weather 호출 예상
    "달러 대비 원화 환율 알려줘",  # → get_exchange_rate 호출 예상
    "오늘 기분 좋은 날이네",       # → 도구 호출 없음 예상
]

print("[Gemini 멀티 함수 테스트]")
for q in queries:
    print(f"\n{'='*60}\n질문: {q}\n{'='*60}")

    chat = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(tools=tools_multi_gemini),
    )
    response = chat.send_message(q)

    if response.function_calls:
        last = None
        for fc in response.function_calls:
            print(f"  → 선택된 함수: {fc.name}({dict(fc.args)})")
            result = available_functions_multi[fc.name](**dict(fc.args))
            last = chat.send_message(
                types.Part.from_function_response(name=fc.name, response={"output": result})
            )
        print(f"  → 최종 답변: {last.text}")
    else:
        print(f"  → 도구 없이 답변: {response.text}")

[Gemini 멀티 함수 테스트]

질문: 서울 날씨 어때?
  → 선택된 함수: get_weather({'city': '서울'})


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

### 5-2. Ollama 멀티 함수 자동 선택 테스트

In [14]:
print("[Ollama 멀티 함수 테스트]")
for q in queries:
    print(f"\n{'='*60}\n질문: {q}\n{'='*60}")

    messages = [{"role": "user", "content": q}]
    response = ollama.chat(model="qwen2.5:3b", messages=messages, tools=tools_multi_ollama)

    if response.message.tool_calls:
        messages.append(response.message)
        for tc in response.message.tool_calls:
            print(f"  → 선택된 함수: {tc.function.name}({tc.function.arguments})")
            result = available_functions_multi[tc.function.name](**tc.function.arguments)
            messages.append({"role": "tool", "content": result})
        final = ollama.chat(model="qwen2.5:3b", messages=messages)
        print(f"  → 최종 답변: {final.message.content}")
    else:
        print(f"  → 도구 없이 답변: {response.message.content}")

[Ollama 멀티 함수 테스트]

질문: 서울 날씨 어때?
  → 선택된 함수: get_weather({'city': '서울'})
  → 최종 답변: 현재 서울의 날씨 정보는 다음과 같습니다:
- 기온: 22℃
- 날씨 상태: 맑음
- 습도: 45%

질문: 달러 대비 원화 환율 알려줘
  → 선택된 함수: get_exchange_rate({'currency': 'KRW'})
  → 최종 답변: 현재 미국 달러(USD) 대비 한국 원화(KRW) 환율은 1 USD당 1380.5 KRW입니다.

질문: 오늘 기분 좋은 날이네
  → 도구 없이 답변: 그런 기분 좋은 하루 되시길 바랍니다! 혹시 도와드릴 일이 있으신가요? 예를 들어, 특정 도시의 날씨 정보를 확인하거나 환율을 알고 싶으실 수도 있습니다. 무엇에 도와드릴 수 있을까요?
